# FPN（PASCAL VOC 2007を用いた物体検出）

---
## 目的
物体検出における代表的な「ネック（neck）」の設計であるFPN（Feature Pyramid Network）を，PASCAL VOC 2007データセットを用いて実装する．バックボーンの深い層（意味的に強いが解像度が低い）と浅い層（解像度は高いが意味的に弱い）を，トップダウン経路（top-down pathway）と横方向接続（lateral connection）によって統合し，全ての解像度レベルが意味的に強い特徴を持つ特徴ピラミッドを構築する仕組みを理解する．検出ヘッド自体は`ssd.ipynb`で実装したSSD形式のデフォルトボックスに基づくヘッドを流用し，「ネックの改善」であるFPNの寄与そのものに焦点を当てる．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCDetection
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.ops import box_iou, box_convert, nms
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．クラス数・データ数などデータセットの詳細は`ssd.ipynb`および`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．読み込み方法（gdownによるダウンロード・`VOCDetectionDataset`・`collate_fn`）も`ssd.ipynb`と同じです．入力画像サイズのみ，本ノートブックのピラミッドの最も深いレベル（stride 64のP6）まで各特徴マップのサイズが割り切れるよう，300ではなく320とします．

In [ ]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
# 背景（物体なし）を0番として，クラスラベルは1〜20を割り当てる
CLASS_TO_IDX = {name: i + 1 for i, name in enumerate(VOC_CLASSES)}
n_class = len(VOC_CLASSES) + 1  # 背景を含めたクラス数
IMG_SIZE = 320

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)


def parse_voc_target(target):
    """VOCDetectionが返すXML由来のdictを，(boxes, labels)のTensorに変換する"""
    objects = target['annotation']['object']
    if isinstance(objects, dict):  # 物体が1つだけの場合はdictのまま返るため，リストに揃える
        objects = [objects]

    boxes, labels = [], []
    for obj in objects:
        bbox = obj['bndbox']
        boxes.append([float(bbox['xmin']), float(bbox['ymin']), float(bbox['xmax']), float(bbox['ymax'])])
        labels.append(CLASS_TO_IDX[obj['name']])

    return {'boxes': torch.tensor(boxes, dtype=torch.float32), 'labels': torch.tensor(labels, dtype=torch.long)}


class VOCDetectionDataset(torch.utils.data.Dataset):
    """VOCDetectionをラップし，画像のリサイズに合わせてボックス座標をスケールしたTensorを返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCDetection(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.to_tensor = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, target = self.voc[idx]
        w, h = image.size
        target = parse_voc_target(target)

        # 元画像サイズ -> (img_size, img_size) へのリサイズに合わせてボックス座標をスケール
        scale = torch.tensor([self.img_size / w, self.img_size / h, self.img_size / w, self.img_size / h])
        target['boxes'] = target['boxes'] * scale

        image = self.to_tensor(image)
        return image, target


def collate_fn(batch):
    # 画像1枚あたりの物体数が異なるため，ボックス・ラベルはリストのまま扱う
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


train_data = VOCDetectionDataset('trainval')
test_data = VOCDetectionDataset('test')

print(len(train_data), len(test_data))
print('クラス数（背景含む）:', n_class)

## FPN（Feature Pyramid Network）
FPNは，物体検出の「ネック」に位置する仕組みで，バックボーンが持つ複数解像度の特徴マップを，トップダウン経路（top-down pathway）と横方向接続（lateral connection）によって統合し，**全ての解像度レベルが意味的に強い特徴を持つ特徴ピラミッド**を構築します．

SSD（`ssd.ipynb`）は，バックボーンの複数の層の出力をそのまま特徴マップとして使用していました．しかし，浅い層（高解像度）はエッジやテクスチャなど低レベルな特徴しか捉えておらず意味的に弱いため，小さな物体の分類には不利です．FPNは，深い層（低解像度・高い意味情報）の特徴を，アップサンプリングしながら浅い層まで伝播させることで，浅い層の特徴マップにも高い意味情報を与え，この問題を解決します．これが，FPNがSSDのような単純な多層特徴マップの利用よりも，特に小さな物体の検出精度を向上させる理由です．

### バックボーン
`ssd.ipynb`と同様に，事前学習済みResNet50の`layer2`・`layer3`・`layer4`の出力を，それぞれC3（stride 8）・C4（stride 16）・C5（stride 32）として取り出します．FPNの核心はこの先のトップダウン経路にあるため，SSDのような追加のダウンサンプリング畳み込み層（`extra1`〜`extra3`）はここでは使用しません．

In [ ]:
class FPNBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool, resnet.layer1)
        self.layer2 = resnet.layer2  # C3, stride  8, channels  512
        self.layer3 = resnet.layer3  # C4, stride 16, channels 1024
        self.layer4 = resnet.layer4  # C5, stride 32, channels 2048
        self.out_channels = [512, 1024, 2048]

    def forward(self, x):
        x = self.stem(x)
        c3 = self.layer2(x)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        return c3, c4, c5

### トップダウン経路と横方向接続
FPNの中核となる処理は次の3ステップです．

1. **横方向接続（lateral connection）**：C3・C4・C5それぞれに1×1畳み込みを適用し，チャネル数を共通の`out_channels`（256）に揃える．
2. **トップダウン経路（top-down pathway）**：最も深いC5の横方向接続結果をP5とする．そこから1つ浅いレベルに向かって，直前のピラミッドレベルを2倍にアップサンプリング（`F.interpolate(..., scale_factor=2, mode='nearest')`）した特徴マップと，そのレベルの横方向接続結果を要素ごとに加算する．これをC3まで繰り返し，P4・P3を得る．
3. **平滑化（smoothing）**：P3・P4・P5それぞれに3×3畳み込みを適用し，アップサンプリング（特に最近傍補間）によって生じるエイリアシング（格子状のノイズ）を軽減する．この3×3畳み込みは原論文でも重要な要素として導入されている．

さらに，より大きな物体を検出するため，P5にstride 2のmax poolingを適用してP6を追加します（原論文の物体検出への応用でも同様の追加レベルが提案されています）．P6はダウンサンプリングのみで生成するため，平滑化の3×3畳み込みは適用しません．

In [ ]:
class FPN(nn.Module):
    """C3・C4・C5から，トップダウン経路と横方向接続により，全レベルが強い意味特徴を持つ特徴ピラミッド(P3〜P6)を構築する"""
    def __init__(self, in_channels_list=(512, 1024, 2048), out_channels=256):
        super().__init__()
        c3_ch, c4_ch, c5_ch = in_channels_list
        # 横方向接続（lateral connection）: チャネル数をout_channelsに揃える1x1畳み込み
        self.lateral3 = nn.Conv2d(c3_ch, out_channels, kernel_size=1)
        self.lateral4 = nn.Conv2d(c4_ch, out_channels, kernel_size=1)
        self.lateral5 = nn.Conv2d(c5_ch, out_channels, kernel_size=1)
        # 3x3畳み込み: アップサンプリングによるエイリアシングを軽減する平滑化
        self.smooth3 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.smooth4 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.smooth5 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        # P6: P5をさらにstride 2でダウンサンプルし，大きな物体向けのレベルを追加する
        self.p6 = nn.MaxPool2d(kernel_size=1, stride=2)
        self.out_channels = out_channels

    def forward(self, c3, c4, c5):
        p5 = self.lateral5(c5)
        p4 = self.lateral4(c4) + F.interpolate(p5, scale_factor=2, mode='nearest')
        p3 = self.lateral3(c3) + F.interpolate(p4, scale_factor=2, mode='nearest')

        p3 = self.smooth3(p3)
        p4 = self.smooth4(p4)
        p5 = self.smooth5(p5)
        p6 = self.p6(p5)

        return [p3, p4, p5, p6]

## デフォルトボックス（Default Box）の生成
デフォルトボックスの生成・エンコード/デコード・マッチング・損失関数の設計は，`ssd.ipynb`で実装したSSDの手法と全く同じです．違いは，特徴マップがSSDの生の（浅い層ほど意味的に弱い）バックボーン出力ではなく，FPNが出力する「全レベルが意味的に強い」ピラミッド（P3〜P6）である点だけです．

本ノートブックのピラミッドはP3（stride 8）・P4（stride 16）・P5（stride 32）・P6（stride 64）の4レベルなので，`ssd.ipynb`の6レベル分の設定から，特徴マップサイズとスケールのスケジュールをこの4レベル用に再計算します．各セルに配置するデフォルトボックスの種類（アスペクト比の集合）は`ssd.ipynb`と同じです．

In [ ]:
FEATURE_MAP_SIZES = [IMG_SIZE // 8, IMG_SIZE // 16, IMG_SIZE // 32, IMG_SIZE // 64]  # P3, P4, P5, P6
ASPECT_RATIOS = [1, 2, 1 / 2, 3, 1 / 3]
MIN_SCALE, MAX_SCALE = 0.1, 0.9
SCALES = [MIN_SCALE + (MAX_SCALE - MIN_SCALE) * i / (len(FEATURE_MAP_SIZES) - 1) for i in range(len(FEATURE_MAP_SIZES))]
SCALES.append(1.0)  # 最後の特徴マップの「1つ先のスケール」として1.0を追加（アスペクト比1の追加ボックス用）


def generate_default_boxes():
    boxes = []
    for k, f in enumerate(FEATURE_MAP_SIZES):
        for i in range(f):
            for j in range(f):
                cx = (j + 0.5) / f
                cy = (i + 0.5) / f

                for ar in ASPECT_RATIOS:
                    w = SCALES[k] * (ar ** 0.5)
                    h = SCALES[k] / (ar ** 0.5)
                    boxes.append([cx, cy, w, h])

                # アスペクト比1で，隣接するスケールとの幾何平均を大きさに持つ追加ボックス
                extra_scale = (SCALES[k] * SCALES[k + 1]) ** 0.5
                boxes.append([cx, cy, extra_scale, extra_scale])

    return torch.tensor(boxes, dtype=torch.float32).clamp(min=0.0, max=1.0)


default_boxes = generate_default_boxes().to(device)  # (num_boxes, 4), (cx, cy, w, h) の正規化座標
default_boxes_xyxy = (box_convert(default_boxes, 'cxcywh', 'xyxy').clamp(0, 1) * IMG_SIZE)  # ピクセル座標(xyxy)
num_boxes_per_cell = len(ASPECT_RATIOS) + 1
print('デフォルトボックスの総数:', default_boxes.shape[0])

### ボックスのエンコード・デコード
学習時のオフセットへのエンコード，推論時の座標へのデコードは，`ssd.ipynb`と全く同じ手法を用います．詳細は`ssd.ipynb`を参照してください．

In [ ]:
def encode_boxes(matched_boxes_xyxy, variances=(0.1, 0.2)):
    """マッチしたGTボックス（xyxy, ピクセル座標）を，デフォルトボックスからのオフセットにエンコードする"""
    matched_cxcywh = box_convert(matched_boxes_xyxy / IMG_SIZE, 'xyxy', 'cxcywh')
    g_cxcy = (matched_cxcywh[:, :2] - default_boxes[:, :2]) / (default_boxes[:, 2:] * variances[0])
    g_wh = torch.log(matched_cxcywh[:, 2:] / default_boxes[:, 2:] + 1e-8) / variances[1]
    return torch.cat([g_cxcy, g_wh], dim=1)


def decode_boxes(loc_preds, variances=(0.1, 0.2)):
    """ネットワークが出力したオフセットを，画像上のボックス座標（xyxy, ピクセル座標）にデコードする"""
    cxcy = loc_preds[..., :2] * variances[0] * default_boxes[:, 2:] + default_boxes[:, :2]
    wh = torch.exp(loc_preds[..., 2:] * variances[1]) * default_boxes[:, 2:]
    boxes_cxcywh = torch.cat([cxcy, wh], dim=-1)
    boxes_xyxy = box_convert(boxes_cxcywh, 'cxcywh', 'xyxy') * IMG_SIZE
    return boxes_xyxy.clamp(min=0, max=IMG_SIZE)  # 画像範囲外に出たボックスをクリップする

### デフォルトボックスとGTボックスのマッチング
マッチング戦略（各GTに対してIoU最大のデフォルトボックスを必ず正例にする + IoUが閾値0.5以上のデフォルトボックスも正例にする，それ以外は背景）も`ssd.ipynb`と同じです．IoUの計算には`torchvision.ops.box_iou`を用います．

In [ ]:
def match_default_boxes(gt_boxes, gt_labels, iou_threshold=0.5):
    """1枚の画像内のGTボックスをデフォルトボックスに割り当て，学習用のラベル・オフセットターゲットを作成する"""
    num_default = default_boxes.shape[0]
    if gt_boxes.shape[0] == 0:
        return torch.zeros(num_default, dtype=torch.long, device=device), torch.zeros(num_default, 4, device=device)

    ious = box_iou(gt_boxes, default_boxes_xyxy)  # (num_gt, num_default)

    # 各デフォルトボックスに対して，最もIoUが高いGTを割り当てる
    best_gt_iou, best_gt_idx = ious.max(dim=0)  # (num_default,)
    # 各GTボックスに対して，最もIoUが高いデフォルトボックスは，閾値によらず必ず正例にする
    _, best_default_idx = ious.max(dim=1)  # (num_gt,)
    best_gt_iou[best_default_idx] = 1.0
    best_gt_idx[best_default_idx] = torch.arange(gt_boxes.shape[0], device=gt_boxes.device)

    labels = gt_labels[best_gt_idx].clone()
    labels[best_gt_iou < iou_threshold] = 0  # 背景

    matched_boxes = gt_boxes[best_gt_idx]
    loc_targets = encode_boxes(matched_boxes)

    return labels, loc_targets

## 損失関数（MultiBox Loss）
損失関数も`ssd.ipynb`と同じ**MultiBox Loss**（正例のデフォルトボックスに対するSmooth L1損失 + 全デフォルトボックスに対する交差エントロピー損失，Hard Negative Miningにより正例:負例が1:`neg_pos_ratio`になるよう負例を選択）を使用します．

FPNをネックとして使う代表的な検出器であるRetinaNetは，このクロスエントロピー損失を，クラス不均衡に強い**Focal Loss**に置き換えることで精度を向上させます．しかし本ノートブックの目的は，FPNのピラミッド構造そのものが多スケール検出に与える効果を確認することにあるため，損失関数は`ssd.ipynb`と同じ重み付き交差エントロピー損失のままとし，Focal Lossは使用しません（Focal Lossは`retinanet.ipynb`で扱います）．

In [ ]:
class MultiBoxLoss(nn.Module):
    def __init__(self, neg_pos_ratio=3):
        super().__init__()
        self.neg_pos_ratio = neg_pos_ratio
        self.smooth_l1 = nn.SmoothL1Loss(reduction='sum')

    def forward(self, loc_preds, conf_preds, loc_targets, labels):
        pos_mask = labels > 0
        num_pos_total = pos_mask.sum().clamp(min=1)

        # Localization Loss: 正例のデフォルトボックスのみで計算
        loc_loss = self.smooth_l1(loc_preds[pos_mask], loc_targets[pos_mask])

        # Confidence Loss: Hard Negative Miningにより，正例:負例が1:neg_pos_ratioになるよう負例を選択
        conf_loss_all = F.cross_entropy(conf_preds.view(-1, conf_preds.size(-1)), labels.view(-1), reduction='none')
        conf_loss_all = conf_loss_all.view(labels.shape)

        conf_loss_pos = conf_loss_all[pos_mask].sum()

        conf_loss_neg = conf_loss_all.clone()
        conf_loss_neg[pos_mask] = -1.0  # 正例は負例マイニングの対象から除外（順位付けで最下位になるようにする）
        num_neg = (pos_mask.sum(dim=1, keepdim=True) * self.neg_pos_ratio).clamp(max=labels.size(1) - 1)
        _, neg_rank = conf_loss_neg.sort(dim=1, descending=True)
        _, neg_rank_idx = neg_rank.sort(dim=1)
        neg_mask = neg_rank_idx < num_neg

        conf_loss_neg_selected = conf_loss_all[neg_mask].sum()

        return (loc_loss + conf_loss_pos + conf_loss_neg_selected) / num_pos_total

## ネットワークの作成
検出ヘッドには，`ssd.ipynb`で定義した`SSDHead`と全く同じ実装を用います（技術は変えず，入力する特徴マップをSSDの生のバックボーン出力からFPNのピラミッド出力に置き換えるだけです）．FPNは全レベルで同じチャネル数（256）に統一されているため，`in_channels_list`は4レベル分すべて256になります．

バックボーン（`FPNBackbone`）・ネック（`FPN`）・検出ヘッド（`SSDHead`）を組み合わせて`FPNDetector`を構築し，`.to(device)`でパラメータをGPU上に配置します．最適化手法は`ssd.ipynb`と同じくモーメンタムSGD（学習率0.001，モーメンタム0.9，weight decay 5e-4）を用います．事前学習済みバックボーンを使用するため，スクラッチ学習のCIFAR100分類モデルよりも小さめの学習率を設定しています．

In [ ]:
class SSDHead(nn.Module):
    """ssd.ipynbのSSDHeadと同じ実装。特徴マップごとに独立な畳み込みでオフセットとクラススコアを出力する"""
    def __init__(self, in_channels_list, num_boxes_per_cell, n_class):
        super().__init__()
        self.loc_layers = nn.ModuleList([
            nn.Conv2d(c, num_boxes_per_cell * 4, kernel_size=3, padding=1) for c in in_channels_list
        ])
        self.conf_layers = nn.ModuleList([
            nn.Conv2d(c, num_boxes_per_cell * n_class, kernel_size=3, padding=1) for c in in_channels_list
        ])
        self.n_class = n_class

    def forward(self, features):
        locs, confs = [], []
        for feat, loc_layer, conf_layer in zip(features, self.loc_layers, self.conf_layers):
            loc = loc_layer(feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, 4)
            conf = conf_layer(feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, self.n_class)
            locs.append(loc)
            confs.append(conf)
        return torch.cat(locs, dim=1), torch.cat(confs, dim=1)


class FPNDetector(nn.Module):
    def __init__(self, n_class, num_boxes_per_cell, fpn_channels=256):
        super().__init__()
        self.backbone = FPNBackbone()
        self.fpn = FPN(self.backbone.out_channels, out_channels=fpn_channels)
        self.head = SSDHead([fpn_channels] * len(FEATURE_MAP_SIZES), num_boxes_per_cell, n_class)

    def forward(self, x):
        c3, c4, c5 = self.backbone(x)
        features = self.fpn(c3, c4, c5)
        return self.head(features)


model = FPNDetector(n_class=n_class, num_boxes_per_cell=num_boxes_per_cell)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)
criterion = MultiBoxLoss(neg_pos_ratio=3)

n_params = sum(p.numel() for p in model.parameters())
print('総パラメータ数:', n_params)

## 学習
学習ループの構造は`ssd.ipynb`と同じです．ミニバッチサイズ8，学習エポック数20とします．画像1枚あたりの物体数が異なるため，`collate_fn`でバッチを作成し，各画像のGTボックスをバッチ内でループしながらデフォルトボックスとマッチングさせます。

In [ ]:
batch_size = 8
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0

    for images, targets in train_loader:
        images = images.to(device)

        batch_labels, batch_loc_targets = [], []
        for t in targets:
            labels, loc_targets = match_default_boxes(t['boxes'].to(device), t['labels'].to(device))
            batch_labels.append(labels)
            batch_loc_targets.append(loc_targets)
        batch_labels = torch.stack(batch_labels)
        batch_loc_targets = torch.stack(batch_loc_targets)

        loc_preds, conf_preds = model(images)

        loss = criterion(loc_preds, conf_preds, batch_loc_targets, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()

    print("epoch: {}, mean loss: {}, elapsed time: {}".format(epoch, sum_loss / len(train_loader), time() - train_start))

## 推論（デコード・NMS）
推論用の関数も`ssd.ipynb`と同じ手順です．出力されたクラススコアにSoftmaxを適用し，クラスごとに信頼度が閾値を超えるデフォルトボックスをデコードしたのち，`torchvision.ops.nms`でクラスごとに重複した検出結果を除去（Non-Maximum Suppression）します。

In [ ]:
def predict(model, image_tensor, conf_threshold=0.5, nms_threshold=0.45):
    model.eval()
    with torch.no_grad():
        loc_preds, conf_preds = model(image_tensor.unsqueeze(0).to(device))
    scores_all = F.softmax(conf_preds[0], dim=-1)
    boxes_all = decode_boxes(loc_preds[0])

    result_boxes, result_scores, result_labels = [], [], []
    for class_idx in range(1, n_class):  # 背景（クラス0）は除外
        class_scores = scores_all[:, class_idx]
        mask = class_scores > conf_threshold
        if mask.sum() == 0:
            continue
        class_boxes = boxes_all[mask]
        class_scores = class_scores[mask]

        keep = nms(class_boxes, class_scores, nms_threshold)
        result_boxes.append(class_boxes[keep])
        result_scores.append(class_scores[keep])
        result_labels.append(torch.full((len(keep),), class_idx, dtype=torch.long))

    if len(result_boxes) == 0:
        return torch.zeros(0, 4), torch.zeros(0), torch.zeros(0, dtype=torch.long)

    return torch.cat(result_boxes).cpu(), torch.cat(result_scores).cpu(), torch.cat(result_labels).cpu()

## テスト
評価指標には，物体検出タスクで広く使われる**mAP（mean Average Precision）**を用います．mAPの算出には`ssd.ipynb`と同様に`torchmetrics`の`MeanAveragePrecision`（`iou_type='bbox'`）を利用し，VOCで伝統的に使われるIoU閾値0.5でのmAP（`map_50`）を確認します。

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False, collate_fn=collate_fn)

metric = MeanAveragePrecision(iou_type='bbox')
model.eval()

for images, targets in test_loader:
    boxes, scores, labels = predict(model, images[0])

    preds = [{'boxes': boxes, 'scores': scores, 'labels': labels}]
    gts = [{'boxes': targets[0]['boxes'], 'labels': targets[0]['labels']}]
    metric.update(preds, gts)

result = metric.compute()
print('mAP@0.5:', result['map_50'].item())
print('mAP@[0.5:0.95]:', result['map'].item())

## 検出結果の可視化
テストデータ1枚に対する検出結果を，`torchvision.utils.draw_bounding_boxes`を用いて画像上に描画します。

In [ ]:
image, target = test_data[0]
boxes, scores, labels = predict(model, image)

# 正規化を戻して0-255のuint8画像に変換
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_show = (image * std + mean).clamp(0, 1)
img_show = (img_show * 255).to(torch.uint8)

label_names = [f'{VOC_CLASSES[l - 1]}: {s:.2f}' for l, s in zip(labels, scores)]
img_with_boxes = draw_bounding_boxes(img_show, boxes, labels=label_names, colors='red', width=2)

plt.figure(figsize=(6, 6))
plt.imshow(img_with_boxes.permute(1, 2, 0))
plt.axis('off')
plt.show()

## オリジナルFPNとの違い
本ノートブックで実装したFPNは，原論文（Lin et al., "Feature Pyramid Networks for Object Detection", 2017）の構成を一部簡略化・変更しています．主な違いは次の通りです．

| | 原論文（FPN） | 本ノートブック |
|---|---|---|
| 検出方式 | RPN・Fast R-CNNなど2段階検出器のneckとして提案 | `ssd.ipynb`のヘッドを流用した1段階（SSD形式）検出器のneckとして使用 |
| ピラミッドレベル | P2〜P6（5レベル，P2はstride 4） | P3〜P6（4レベル，最も浅いP2は省略し計算量を抑制） |
| バックボーン | ResNet50/101（ImageNet事前学習済み） | ResNet50（ImageNet事前学習済み，変更なし） |
| 各レベルのチャネル数 | 全レベル256チャネルに統一 | 全レベル256チャネルに統一（変更なし） |
| P6の生成方法 | P5に対するstride 2のmax pooling | 同じ（変更なし） |
| anchor/default box設計 | （検出器側の設計に依存，RPNなら1セルあたり3スケール×3アスペクト比の9種類など） | `ssd.ipynb`と同じ6種類（5アスペクト比＋1）に統一 |
| 分類損失 | （検出器側の設計に依存） | 重み付き交差エントロピー損失＋Hard Negative Mining．Focal Lossは使用しない（Focal Lossは`retinanet.ipynb`で扱う） |

ピラミッドの構築方法（横方向接続・トップダウン経路・3×3平滑化畳み込み）自体は原論文と同じですが，2段階検出器ではなく`ssd.ipynb`のSSD形式ヘッドと組み合わせて1段階検出器として使用している点，およびP2レベルを省略している点が主な変更点です。

## 課題

1. ピラミッドレベル数（P3〜P6を，P3〜P5のみに減らす，あるいはP2を追加した5レベルにする）を変更し，パラメータ数や検出精度，特に小さい物体の検出への影響を確認しましょう．

2. `FPN`の平滑化畳み込み（`smooth3`・`smooth4`・`smooth5`）を取り除いた場合と比較し，アップサンプリングによるエイリアシングが検出精度にどう影響するか確認しましょう．

3. 本ノートブックの検出結果を，`ssd.ipynb`で同じテスト画像に対して得られた検出結果と比較し，特に小さい物体の検出において，FPNのピラミッド構造がどのように寄与しているか確認しましょう．